In [ ]:
import sys
sys.path.append('../../src/')

import matplotlib.pyplot as plt

# Import methods
import QGP.Defocusing.Methods as Def
import QGP.Focusing.Methods as Foc
from QGP.Soliton_Gen import *
from Utilities import *

import config

In [ ]:
def _get_valid_refpoints(xmin, xmax):
    d = xmax - xmin
    ref1 = xmin + d / 4.
    ref2 = xmax - d / 4.
    return ref1, ref2

In [ ]:
# Test energy is well calculated
def A(c, k):
    return 3*c**4*k**2 - 8*c**2*k**2 - 2*c**2*k + 8*k - 1

def B(c, k):
    return 3*c**2*k - 4*k - 1

def C(c, k):
    return c**2*k - 4*k - 1

def D(c, k):
    return np.sqrt((1 - 2*k) * (2 - c**2))

def L_c(c, k):
    return np.sqrt((2 - c**2) / (1 - 2*k))

# s = 1 for D_1 and s = -1 for D_3
def E_D_1_3(c, k, s):
    return (A(c, k) * np.arctan(np.sqrt(k) * L_c(c, k)) - s * np.sqrt(k) * B(c, k) * D(c, k)) / (16 * k**(3/2))

def E_D_2(c, k):
    return -(A(c, k) * np.arctanh(np.sqrt(np.abs(k)) * L_c(c, k))- np.sqrt(np.abs(k)) * B(c, k) * D(c, k)) / (16 * np.abs(k)**(3/2))


In [ ]:
def soliton_snapshot(x, ws, nplots, figsize=(6, 7), Ls=None, titles=None, save_path=None):
    fig, axes = plt.subplots(1, nplots, figsize=figsize, sharey=True)

    if nplots == 1:
        axes = [axes]

    if Ls is not None:
        for j in range(len(axes)):
            ax = axes[j]
            L = Ls[j]
            ax.set_xlim(-L, L)

    if titles is not None:
        for j in range(len(axes)):
            ax = axes[j]
            title = titles[j]
            ax.set_title(title)

    for j in range(len(axes)):
        ax = axes[j]
        w = ws[j]

        ax.plot(x, np.abs(w), color='C0', lw=2.8, ms=0, zorder=5)   # azul
        ax.plot(x, -np.abs(w), color='C0', lw=1, ms=0)   # azul
        ax.plot(x, np.real(w),   color='green', lw=1, ms=0, label="Re")   # rojo
        ax.plot(x, np.imag(w),   color='C3', lw=1, ms=0, label="Im")   # verde

        ax.set_ylim(
            min(np.min(np.real(w)), np.min(np.imag(w)), np.min(-np.abs(w))) - 0.2,
            max(np.max(np.real(w)), np.max(np.imag(w)), np.max(np.abs(w))) + 0.2
        )

        ax.tick_params(direction='in', top=True, right=True)

        #if i < nplots - 1:
        #    ax.tick_params(labelbottom=False)

        ax.set_xlabel(r'$x$', fontsize=10)
    axes[0].set_ylabel(r'$|\Psi|$', fontsize=10, rotation="horizontal", labelpad=10)

    plt.legend(fontsize=8)
    plt.subplots_adjust(wspace=0.08)

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
L = 100
w_1, x = soliton(np.sqrt(2)/2, 1/4, soliton_type="dark", L=L, N=1001)
w_2, x = soliton(0.6, -50, soliton_type="dark", L=L, N=1001)
w_3, x = soliton(0.4, -50, soliton_type="dark", L=L, N=1001)

In [ ]:
ws = [w_1, w_2, w_3]
Ls = [10, 60, 60]
titles = [r"$c=\sqrt{2}/2$, $\upkappa=-1/4$", r"$c=0.6$, $\upkappa=50$", r"$c=0.4$, $\upkappa=50$"]
soliton_snapshot(x, ws, 3, figsize=(8, 2), Ls=Ls, titles=titles, save_path="Solitons/darks.pdf")

# Dark Soliton in $D_1$

In [ ]:
path_save = "Solitons/Dark_D1/"

In [ ]:
# Function to invert

def F_c_k(y, c, k):
    
    # Domain
    ymin = 0
    ymax = 1-c*c/2
    
    y = np.asarray(y)
    
    valid_mask = (y > ymin) & (y <= ymax)
    
    # Assign infinite to values outside the domain
    val = np.full_like(y, np.inf, dtype=np.float64)
    val[y > ymax] = -np.inf
    
    def f_c_k(y):
        sum_1 = 2 * np.sqrt(k) * np.arctan(np.sqrt(k) * np.sqrt((2-c*c-2*y)/(1-2*k+2*k*y)))
        sum_2 = 2 * np.sqrt((1-2*k)/(2-c*c)) * np.arctanh(np.sqrt(((1-2*k)*(2-c*c-2*y))/((2-c*c)*(1-2*k+2*k*y))))
        output = sum_1 + sum_2
        return output
    
    val[valid_mask] = f_c_k(y[valid_mask])
    
    return val

In [ ]:
### Varying k ###

N = 6
c = np.sqrt(2)/2
ks = np.linspace(0, 1/2, N, endpoint=False)[1:]

y_to_plot = np.linspace(0, 1-c*c/2, 1_000)

for k in ks:
    print("c =", c, "k =", k)
    plt.plot(y_to_plot, F_c_k(y_to_plot, c, k), label=r"$(c, \upkappa)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$y$")
plt.ylabel(r"$G_{c, \upkappa}(y)$")

plt.savefig(path_save+"Func_to_inv_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying c ###

N = 5
cs = np.linspace(0, np.sqrt(2), N, endpoint=False)
k = 1/4

for c in cs:
    print("c =", c, "k =", k)
    y_to_plot = np.linspace(0, 1-c*c/2, 1_000)[1:]
    plt.plot(y_to_plot, F_c_k(y_to_plot, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()
    
plt.xlabel(r"$y$")
plt.ylabel(r"$G_{c, k}(y)$")
plt.savefig(path_save+"Func_to_inv_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## Dark soliton

In [ ]:
import warnings

## eta_c_k generator

def eta_c_k(x, c, k, x_0=0, even=True):
    
    # Domain
    ymin = 0
    ymax = 1-(c*c/2)
    
    ref1, ref2 = _get_valid_refpoints(ymin, ymax)
    
    min_kwargs = {}
    min_kwargs['bracket'] = (ref1, ref2)
    min_kwargs['tol'] = 1.48e-12
    min_kwargs['method'] = 'Golden'
    
    x = np.asarray(x)
    N = x.size
    m = N//2
    
    xpos = x[N//2:]
    
    F_to_inv = lambda y: F_c_k(y, c, k)-x_0
    
    results = np.full_like(x, np.nan, dtype=np.float64)
    resultsmask = np.zeros(x. shape, dtype=bool)
    
    for j in range(xpos.size):
        
        if F_to_inv(ymax) == xpos[j]: #is_close
            results[N//2+j] = ymax
        
        if F_to_inv(ymin) == xpos[j]:
            results[N//2+j] = ymin
        
        optimizer = (lambda x, j=j,
                        bounded_f=F_to_inv: (np.abs(F_to_inv(x) - xpos[j]))**2)
        
        try:
            with warnings.catch_warnings(record=True):
                result = minimize_scalar(optimizer, **min_kwargs)

            results[N//2+j] = result.x
            resultsmask[N//2+j] = result.success

        except:
            resultsmask[N//2+j] = False

    if any(~resultsmask[N//2:]):
            warnings.warn("Trouble calculating inverse for values: "
                          "%s" % str(xpos[~resultsmask[N//2:]]), RuntimeWarning)   
    try:
        np.testing.assert_array_almost_equal(xpos, F_to_inv(results[N//2:]), decimal=2)

    except AssertionError:
        warnings.warn("Results obtained with less than %g "
                          "decimal digits of accuracy"
                          % 2, RuntimeWarning)

    if even:
        if N%2 == 0:
            results[:N//2] = np.flip(results[N//2:])
        else:
            results[:N//2] = np.flip(results[N//2+1:])
        return results
    
    return results[N//2:]


In [ ]:
#import warnings
#warnings.filterwarnings("ignore")

### Invert function ###

### Varying k ###

N = 5
c = np.sqrt(2)/2
ks = np.linspace(0, 1/2, N, endpoint=False)[1:]

L = 10
x_0 = 0
x = np.linspace(-L, L, 1_001)-x_0

for k in ks:
    print("c =", c, "k =", k)
    plt.plot(x, eta_c_k(x, c, k, x_0=x_0), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
             marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")

plt.savefig(path_save+"eta_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
#import warnings
#warnings.filterwarnings("ignore")

### Invert function ###

### Varying k ###

N = 4
cs = np.linspace(0, np.sqrt(2), N, endpoint=False)
k = 1/4

L = 10
x_0 = 0
x = np.linspace(-L, L, 1_001)-x_0

for c in cs:
    print("c =", c, "k =", k)
    plt.plot(x, eta_c_k(x, c, k, x_0=x_0), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))), 
             marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")

plt.savefig(path_save+"eta_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
print("c = ", c, "k = ", k)
Ns = np.array([2**(j) for j in range(2, 11)])
hs = []
errors = [] #np.zeros(len(Ns))
xs = []
L = 5

for j in range(len(Ns)):
    N = Ns[j]
    x = np.linspace(-L, L, N)
    eta = eta_c_k(x, c, k, even=False)
    
    x_pos = x[N//2:]
    h = x_pos[1] - x_pos[0]
    hs += [h]
    F_eta = F_c_k(eta, c, k)
    error = np.abs(x_pos - F_eta)
    
    errors += [error]
    xs += [x_pos]
    
hs = np.array(hs)

In [ ]:
for j in range(len(errors)):
    plt.plot(xs[j], errors[j], ".", label="Ns = {}".format(Ns[j]))
#plt.plot(hs, hs**-0.8 * np.exp(-28))
plt.legend(fontsize=10)
plt.hlines(1.48e-12, 0, 5)
plt.yscale("log")
#plt.xscale("log")
plt.show()

In [ ]:
print("c = ", c, "k = ", k)
Ns = np.array([2**(j) for j in range(2, 11)])
hs = []
errors = [] #np.zeros(len(Ns))
ys = []
L = 5

for j in range(len(Ns)):
    N = Ns[j]
    y = np.linspace(0, 1-c*c/2, N)[1:]
    F_ = F_c_k(y, c, k)
    F_ = np.append(np.flip(F_), F_)
    eta = eta_c_k(F_, c, k, even=False)
    error = np.abs(y - eta)
    
    errors += [error]
    ys += [y]
    
hs = np.array(hs)

In [ ]:
for j in range(len(errors)):
    plt.plot(ys[j], errors[j], ".", label="Ns = {}".format(Ns[j]))
#plt.plot(hs, hs**-0.8 * np.exp(-28))
plt.legend(fontsize=10)
#plt.hlines(1.48e-12, 0, 5)
#plt.yscale("log")
#plt.xscale("log")
plt.show()

In [ ]:
### Varying c ###

N = 4
cs = np.linspace(0, np.sqrt(2), N, endpoint=False)
k = 1/4
print("c =", cs, "k =", k)

L = 5
x = np.linspace(-L, L, 1_000)

for c in cs:
    
    plt.plot(x, eta_c_k(x, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))))
    plt.legend()

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")
plt.show()

## Checking ODE $\eta$ 

In [ ]:
c = np.sqrt(2)/2
k = 1/4

L = 10
x = np.linspace(-L, L, 10_000)
h = x[1] - x[0]

eta = eta_c_k(x, c, k)
eta_prime = np.gradient(eta, h)
eta_prime_prime = np.gradient(eta_prime, h)

ODE = (1 - 2 * k + 2 * k * eta) * eta_prime_prime + k * eta_prime**2 + 3 * eta**2 - (2-c**2) * eta
ODE_2 = (1 - 2 * k + 2 * k * eta) * eta_prime**2 - (2 - c**2 - 2 * eta) * eta**2

In [ ]:
plt.plot(x, ODE, ".")
plt.plot(x, ODE_2, ".")

## $u$ variable

In [ ]:
## Solitons in variable u

def rho_c_k(x, eta):
    output = np.sqrt(1 - eta)
    return output
    

def theta_c_k(x, c, eta):
    
    ## Function to integrate
    integrand = eta/(1 - eta)
    
    N = x.size
    m = N//2
    xpos = x[m:]
    theta = np.zeros(N)
    
    dx = xpos[1] - xpos[0]

    for j in range(xpos.size-1):
        
        theta[j+1+m] = theta[j+m] + c/2 * (dx/2 * (integrand[j+m] + integrand[j+m+1]))
        
    if N%2 == 0:
        theta[:m] = -np.flip(theta[m:])

    else:
        theta[:m] = -np.flip(theta[m+1:])
    
    return theta


def u_c_k(x, c, eta):
    i = 1j
    
    if c == 0:
        output = np.sign(x) * np.sqrt(1 - eta)
        return output
    
    rho = rho_c_k(x, eta)
    theta = theta_c_k(x, c, eta)
    return rho * np.exp(i*theta)

def norm(x):
    return np.abs(x)**2

In [ ]:
### Varying k ###

N = 5
c = np.sqrt(2)/2 #0
ks = np.linspace(0, 1/2, N, endpoint=False)[1:]


L = 5
#x = np.linspace(-L, L, 2_001)


for k in ks:
    print("c =", c, "k =", k)
    #eta = eta_c_k(x, c, k)
    u_c_k, x = soliton(c, k, soliton_type="dark", L=L, N=1001)

    plt.plot(x, norm(u_c_k), label=r"$\upkappa = {}$".format(round(-k, 3)), #r"$(c, k)=\Big(\frac{\sqrt{2}}{2},$"+str(round(k, 3))+r"$\Big)$",
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u(x)|^2$", rotation='horizontal', ha='right', multialignment='center')
plt.savefig(path_save+"D_1_dark_soliton_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"D_1_dark_soliton_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying c ###

N = 4
cs = np.linspace(0, np.sqrt(2), N, endpoint=False)
k = 1/4
print("c =", cs, "k =", k)

labels = ["0", r"$\frac{\sqrt{2}}{4}$", r"$\frac{\sqrt{2}}{2}$", r"$\frac{3\sqrt{2}}{4}$"]

L = 5
#x = np.linspace(-L, L, 2_001)

for j in range(len(cs)):
    c = cs[j]
    #eta = eta_c_k(x, c, k)
    u_c_k, x = soliton(c, k, soliton_type="dark", L=L, N=1001)
    plt.plot(x, norm(u_c_k), label=r"$c = $"+labels[j],#r"$(c, k)=\Big($"+labels[j]+r"$,$"+str(round(k, 3))+r"$\Big)$",
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u(x)|^2$", rotation='horizontal', ha='right', multialignment='center')

plt.savefig(path_save+"D_1_dark_soliton_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"D_1_dark_soliton_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## ODE $\theta$

In [ ]:
c = np.sqrt(2)/2
k = 1/4

L = 10
x = np.linspace(-L, L, 10_001)
h = x[1] - x[0]

eta = eta_c_k(x, c, k)

theta = theta_c_k(x, c, eta)
theta_prime = np.gradient(theta, h, edge_order=2)
i = 1j

ODE_theta = theta_prime - c/2 * eta/(1 - eta)

In [ ]:
plt.plot(x, theta, ".")

In [ ]:
plt.plot(x, ODE_theta, ".")

In [ ]:
eta[len(eta)//2], 1-c**2/2

## ODE $u$

In [ ]:
c = np.sqrt(2)/2
k = 1/4

L = 10
x = np.linspace(-L, L, 10_000)
h = x[1] - x[0]

eta = eta_c_k(x, c, k)

u = u_c_k(x, c, eta)
u_prime = np.gradient(u, h)
u_prime_prime = np.gradient(u_prime, h)
norm_prime = np.gradient(1-np.abs(u)**2, h)
norm_prime_prime = np.gradient(norm_prime, h)
i = 1j

ODE_u = i * c * u_prime + u_prime_prime + u * (1 - np.abs(u)**2) + k * u * norm_prime_prime

In [ ]:
plt.plot(norm(u))

In [ ]:
plt.plot(x, np.real(ODE_u), ".")

In [ ]:
plt.plot(x, np.imag(ODE_u), ".")

# Dark Soliton in $D_2$

In [ ]:
path_save = "Solitons/Dark_D2/"

In [ ]:
# Function to invert
def G_c_k(y, c, k):
    
    # Domain
    ymin = 0
    ymax = 1-c*c/2
    
    y = np.asarray(y)
    
    valid_mask = (y > ymin) & (y <= ymax)
    
    # Assign infinite to values outside the domain
    val = np.full_like(y, np.inf, dtype=np.float64)
    val[y > ymax] = -np.inf
    
    def g_c_k(y):
        root = np.sqrt((2-c*c-2*y)/(1-2*k+2*k*y))
        sum_1 = -2 * np.sqrt(-k) * np.arctanh(np.sqrt(-k) * root)
        sum_2 = 2 * np.sqrt((1-2*k)/(2-c*c)) * np.arctanh(np.sqrt((1-2*k)/(2-c*c)) * root)
        output = sum_1 + sum_2
        return output
    
    val[valid_mask] = g_c_k(y[valid_mask])
    
    return val

In [ ]:
### Varying k ###

N = 5
c = np.sqrt(2)/2
ks = -np.linspace(0, 20, N)[1:]
print("c =", c, "k =", ks)

y_to_plot = np.linspace(0, 1-c*c/2, 1_000)

for k in ks:
    plt.plot(y_to_plot, G_c_k(y_to_plot, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$y$")
plt.ylabel(r"$G_{c, k}(y)$")

#plt.savefig(path_save+"Func_to_inv_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying c ###

N = 5
cs = np.linspace(0, np.sqrt(2), N, endpoint=False)
k = -10
print("c =", cs, "k =", k)

for c in cs:
    y_to_plot = np.linspace(0, 1-c*c/2, 1_000)[1:]
    plt.plot(y_to_plot, G_c_k(y_to_plot, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()
    
plt.xlabel(r"$y$")
plt.ylabel(r"$G_{c, k}(y)$")

#plt.savefig(path_save+"Func_to_inv_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## Dark soliton

In [ ]:
def _get_valid_refpoints(xmin, xmax):
    d = xmax - xmin
    ref1 = xmin + d / 4.
    ref2 = xmax - d / 4.
    return ref1, ref2

In [ ]:
import warnings

## eta_c_k generator

def etaG_c_k(x, c, k, even=True):
    
    # Domain
    ymin = 0
    ymax = 1-(c*c/2)
    
    ref1, ref2 = _get_valid_refpoints(ymin, ymax)
    
    min_kwargs = {}
    min_kwargs['bracket'] = (ref1, ref2)
    min_kwargs['tol'] = 1.48e-12
    min_kwargs['method'] = 'Golden'
    
    x = np.asarray(x)
    N = x.size
    m = N//2
    
    xpos = x[N//2:]
    
    F_to_inv = lambda y: G_c_k(y, c, k)
    
    results = np.full_like(x, np.nan, dtype=np.float64)
    resultsmask = np.zeros(x. shape, dtype=bool)
    
    for j in range(xpos.size):
        
        if F_to_inv(ymax) == xpos[j]: #is_close
            results[N//2+j] = ymax
        
        if F_to_inv(ymin) == xpos[j]:
            results[N//2+j] = ymin
        
        optimizer = (lambda x, j=j,
                        bounded_f=F_to_inv: (np.abs(F_to_inv(x) - xpos[j]))**2)
        
        try:
            with warnings.catch_warnings(record=True):
                result = minimize_scalar(optimizer, **min_kwargs)

            results[N//2+j] = result.x
            resultsmask[N//2+j] = result.success

        except:
            resultsmask[N//2+j] = False

    if any(~resultsmask[N//2:]):
            warnings.warn("Trouble calculating inverse for values: "
                          "%s" % str(xpos[~resultsmask[N//2:]]), RuntimeWarning)   
    try:
        np.testing.assert_array_almost_equal(xpos, F_to_inv(results[N//2:]), decimal=2)

    except AssertionError:
        warnings.warn("Results obtained with less than %g "
                          "decimal digits of accuracy"
                          % 2, RuntimeWarning)

    if even:
        if N%2 == 0:
            results[:N//2] = np.flip(results[N//2:])
        else:
            results[:N//2] = np.flip(results[N//2+1:])
        return results
    
    return results[N//2:]


In [ ]:
#import warnings
#warnings.filterwarnings("ignore")

### Invert function ###

### Varying k ###

N = 5
c = np.sqrt(2)/2
ks = -np.linspace(0, 20, N)[1:]
print("c =", c, "k =", ks)

L = 30
x = np.linspace(-L, L, 1_001)

for k in ks:
    
    plt.plot(x, etaG_c_k(x, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
             marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")

#plt.savefig(path_save+"eta_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
print("c = ", c, "k = ", k)
Ns = np.array([2**(j) for j in range(2, 11)])
hs = []
errors = [] #np.zeros(len(Ns))
xs = []
L = 5

for j in range(len(Ns)):
    N = Ns[j]
    x = np.linspace(-L, L, N)
    eta = etaG_c_k(x, c, k, even=False)
    
    x_pos = x[N//2:]
    h = x_pos[1] - x_pos[0]
    hs += [h]
    F_eta = G_c_k(eta, c, k)
    error = np.abs(x_pos - F_eta)
    
    errors += [error]
    xs += [x_pos]
    
hs = np.array(hs)

In [ ]:
for j in range(len(errors)):
    plt.plot(xs[j], errors[j], ".", label="Ns = {}".format(Ns[j]), ms=1)
#plt.plot(hs, hs**-0.8 * np.exp(-28))
plt.legend(fontsize=5)
plt.hlines(1.48e-12, 0, 5)
plt.yscale("log")
#plt.xscale("log")
plt.show()

In [ ]:
print("c = ", c, "k = ", k)
Ns = np.array([2**(j) for j in range(2, 11)])
hs = []
errors = [] #np.zeros(len(Ns))
ys = []
L = 5

for j in range(len(Ns)):
    N = Ns[j]
    y = np.linspace(0, 1-c*c/2, N)[1:]
    F_ = G_c_k(y, c, k)
    F_ = np.append(np.flip(F_), F_)
    eta = etaG_c_k(F_, c, k, even=False)
    error = np.abs(y - eta)
    
    errors += [error]
    ys += [y]
    
hs = np.array(hs)

In [ ]:
for j in range(len(errors)):
    plt.plot(ys[j], errors[j], ".", label="Ns = {}".format(Ns[j]), ms=1)
#plt.plot(hs, hs**-0.8 * np.exp(-28))
plt.legend(fontsize=5)
#plt.hlines(1.48e-12, 0, 5)
#plt.yscale("log")
#plt.xscale("log")
plt.show()

In [ ]:
### Varying c ###

N = 4
cs = np.linspace(0, np.sqrt(2), N, endpoint=False)
k = -5
print("c =", cs, "k =", k)

L = 20
x = np.linspace(-L, L, 1_000)

for c in cs:
    
    plt.plot(x, etaG_c_k(x, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))), 
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")

#plt.savefig(path_save+"eta_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## Checking ODE $\eta$ 

In [ ]:
from numpy.linalg import norm as L2_norm

#P'(c)
def P_p(c, k):
    sqrt = np.sqrt((1 - 2 * k)/(2 - c**2))
    frac = (3 * c**2 * k - 4 * k - 1)/(4 * np.sqrt(-k))
    
    sum_1 = frac * np.arctanh(np.sqrt(-k)/sqrt)
    sum_2 = 3 * (2 - c**2)/4 * sqrt
    output = sum_1 - sum_2
    return output

#P''(c)
def P_pp(c, k):
    frac = 6 * c * k/(4 * np.sqrt(-k))
    sqrt = np.sqrt((1 - 2 * k)/(2 - c**2))
    sum_1 = frac * np.arctanh(np.sqrt(-k)/sqrt)
    
    frac_2 = (3 * c**2 * k - 4 * k - 1)/4
    sum_2 = frac_2 * sqrt * c/(1 - k * c**2) + 3/4 * c * sqrt
    output = sum_1 - sum_2
    return output

def newton_1D(x_0, f, f_prime, tol=1e-8):
    x_k = x_0
    error = np.add.reduce(np.abs(f(x_k)))
    while error > tol:
        f_1 = -f(x_k)
        f_2 = f_prime(x_k)
        delta_x = f_1/f_2
        x_k_1 = x_k + delta_x
        error = L2_norm([x_k - x_k_1], ord=2) #np.add.reduce(np.abs(f(x_k)))
        x_k = x_k_1

    if error != error:
        return 0, 1

    return x_k, error

In [ ]:
c_0 = np.sqrt(2)/2
k = -20

P_p_ = lambda c: P_p(c, k)
P_pp_ = lambda c: P_pp(c, k)

c_star, error = newton_1D(c_0, P_p_, P_pp_)
c = c_star - 0.1

L = 80
x = np.linspace(-L, L, 40_000)
h = x[1] - x[0]

eta = etaG_c_k(x, c, k)
eta_prime = np.gradient(eta, h)
eta_prime_prime = np.gradient(eta_prime, h)

ODE = (1 - 2 * k + 2 * k * eta) * eta_prime_prime + k * eta_prime**2 + 3 * eta**2 - (2-c**2) * eta
ODE_2 = (1 - 2 * k + 2 * k * eta) * eta_prime**2 - (2 - c**2 - 2 * eta) * eta**2

In [ ]:
plt.plot(x, ODE, ls="-", marker="none")
plt.plot(x, ODE_2, ls="-", marker="none")

## $u$ variable

In [ ]:
## Solitons in variable u

def rho_c_k(x, eta):
    output = np.sqrt(1 - eta)
    return output
    

def theta_c_k(x, c, eta):
    
    ## Function to integrate
    integrand = eta/(1 - eta)
    
    N = x.size
    m = N//2
    xpos = x[m:]
    theta = np.zeros(N)
    
    dx = xpos[1] - xpos[0]

    for j in range(xpos.size-1):
        
        theta[j+1+m] = theta[j+m] + c/2 * (dx/2 * (integrand[j+m] + integrand[j+m+1]))
    
    theta[:m] = -np.flip(theta[m+1:])
    
    return theta


def u_c_k(x, c, eta):
    i = 1j
    
    if c == 0:
        output = np.sign(x) * np.sqrt(1 - eta)
        return output
    
    rho = rho_c_k(x, eta)
    theta = theta_c_k(x, c, eta)
    return rho * np.exp(i*theta)

def norm(x):
    return np.abs(x)**2

In [ ]:
### Varying k ###

N = 5
c = 0.1 #np.sqrt(2)/2 #0
ks = -np.linspace(0, 20, N)[1:]
print("c =", c, "k =", ks)


L = 30
x = np.linspace(-L, L, 2_001)


for k in ks:
    print("c =", c, "k =", k)
    #eta = eta_c_k(x, c, k)
    u_c_k, x = soliton(c, k, soliton_type="dark", L=L, N=1001)

    plt.plot(x, norm(u_c_k), label=r"$\upkappa = {}$".format(round(-k, 3)), #r"$(c, k)=\Big(\frac{\sqrt{2}}{2},$"+str(round(k, 3))+r"$\Big)$",
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u(x)|^2$", rotation='horizontal', ha='right', multialignment='center')
plt.savefig(path_save+"D_2_dark_soliton_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"D_1_dark_soliton_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying c ###

N = 4
cs = np.linspace(0, np.sqrt(2), N, endpoint=False)
k = -5
print("c =", cs, "k =", k)

labels = ["0", r"$\frac{\sqrt{2}}{4}$", r"$\frac{\sqrt{2}}{2}$", r"$\frac{3\sqrt{2}}{4}$"]
L = 30
#x = np.linspace(-L, L, 2_001)

for j in range(len(cs)):
    c = cs[j]
    #eta = eta_c_k(x, c, k)
    u_c_k, x = soliton(c, k, soliton_type="dark", L=L, N=1001)
    plt.plot(x, norm(u_c_k), label=r"$c = $"+labels[j],#r"$(c, k)=\Big($"+labels[j]+r"$,$"+str(round(k, 3))+r"$\Big)$",
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u(x)|^2$", rotation='horizontal', ha='right', multialignment='center')

plt.savefig(path_save+"D_2_dark_soliton_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"D_1_dark_soliton_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## ODE $\theta$

In [ ]:
c_0 = np.sqrt(2)/2
k = -50

P_p_ = lambda c: P_p(c, k)
P_pp_ = lambda c: P_pp(c, k)

c, error = newton_1D(c_0, P_p_, P_pp_)

L = 80
x = np.linspace(-L, L, 10_001)
h = x[1] - x[0]

eta = etaG_c_k(x, c, k)

theta = theta_c_k(x, c, eta)
theta_prime = np.gradient(theta, h, edge_order=2)
i = 1j

ODE_theta = theta_prime - c/2 * eta/(1 - eta)

In [ ]:
plt.plot(x, theta, ".")

In [ ]:
plt.plot(x, ODE_theta, ".", ms=1)

## ODE $u$

In [ ]:
c = np.sqrt(2)/2
k = -50

L = 80
x = np.linspace(-L, L, 10_001)
h = x[1] - x[0]

eta = etaG_c_k(x, c, k)

u = u_c_k(x, c, eta)
u_prime = np.gradient(u, h)
u_prime_prime = np.gradient(u_prime, h)
norm_prime = np.gradient(1-np.abs(u)**2, h)
norm_prime_prime = np.gradient(norm_prime, h)
i = 1j

ODE_u = i * c * u_prime + u_prime_prime + u * (1 - np.abs(u)**2) + k * u * norm_prime_prime

In [ ]:
plt.plot(norm(u), marker="none", ls="-")

In [ ]:
plt.plot(x, np.real(ODE_u), ".", ms=1)

In [ ]:
plt.plot(x, np.imag(ODE_u), ".", ms=1)

# Cuspons in $D_3$

In [ ]:
def f_c_k(y, c, k):
    # Domain
    ymin = 0
    ymax = 1 - 1 / (2 * k)
    
    y = np.asarray(y)
    
    valid_mask = (y > ymin) & (y < ymax)
    
    # Assign infinite to values outside the domain
    val = np.full_like(y, np.inf, dtype=np.float64)
    val[y >= ymax] = 0 #np.inf
    val[y < ymin] = np.inf
    
    def f_c_k(y):
        term1 = 2 * np.sqrt(k) * np.arctan(np.sqrt(k) * np.sqrt((2 - c**2 - 2*y) / (1 - 2*k + 2*k*y)))
        term2 = 2 * np.sqrt((1 - 2*k) / (2 - c**2)) * np.arctanh(np.sqrt(((2 - c**2) * (1 - 2*k + 2*k*y)) / ((1 - 2*k) * (2 - c**2 - 2*y))))
        term3 = np.pi * np.sqrt(k)

        return term1 + term2 - term3

    val[valid_mask] = f_c_k(y[valid_mask])

    return val

In [ ]:
### Varying k ###

N = 6
c = 2
ks = np.linspace(1/2, 4, N)[1:]

for k in ks:
    y_max = 1 - 1 / (2 * k)

    y_to_plot = np.linspace(0, y_max, 1_000)[1:]
    
    print("c =", c, "k =", k)
    plt.plot(y_to_plot, f_c_k(y_to_plot, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$y$")
plt.ylabel(r"$f_{c, k}(y)$")

#plt.savefig(path_save+"Func_to_inv_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying k ###

N = 6
k = 0.6

y_max = 1 - 1 / (2 * k)
y_to_plot = np.linspace(0, y_max, 1_000)[1:]

cs = np.linspace(1.5, 5, N)[1:]

for c in cs:
    print("c =", c, "k =", k)
    plt.plot(y_to_plot, f_c_k(y_to_plot, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$y$")
plt.ylabel(r"$f_{c, k}(y)$")

#plt.savefig(path_save+"Func_to_inv_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## Cuspons

In [ ]:
import warnings

## eta_c_k generator

def eta_f_c_k(x, c, k, x_0=0, even=True, eps_smooth=None):
    
    # Domain
    ymin = 0
    ymax = 1 - 1 / (2 * k)
    
    ref1, ref2 = _get_valid_refpoints(ymin, ymax)
    
    min_kwargs = {}
    min_kwargs['bracket'] = (ref1, ref2)
    min_kwargs['tol'] = 1.48e-12
    min_kwargs['method'] = 'Golden'
    
    x = np.asarray(x)
    if eps_smooth:
        x = np.sqrt(x**2 + eps_smooth)

    N = x.size
    m = N//2
    
    xpos = x[N//2:]
    
    F_to_inv = lambda y: f_c_k(y, c, k)-x_0
    
    results = np.full_like(x, np.nan, dtype=np.float64)
    resultsmask = np.zeros(x. shape, dtype=bool)
    
    for j in range(xpos.size):
        
        if F_to_inv(ymax) == xpos[j]: #is_close
            results[N//2+j] = ymax
        
        if F_to_inv(ymin) == xpos[j]:
            results[N//2+j] = ymin
        
        optimizer = (lambda x, j=j,
                        bounded_f=F_to_inv: (np.abs(F_to_inv(x) - xpos[j]))**2)
        
        try:
            with warnings.catch_warnings(record=True):
                result = minimize_scalar(optimizer, **min_kwargs)

            results[N//2+j] = result.x
            resultsmask[N//2+j] = result.success

        except:
            resultsmask[N//2+j] = False

    if any(~resultsmask[N//2:]):
            warnings.warn("Trouble calculating inverse for values: "
                          "%s" % str(xpos[~resultsmask[N//2:]]), RuntimeWarning)   
    try:
        np.testing.assert_array_almost_equal(xpos, F_to_inv(results[N//2:]), decimal=2)

    except AssertionError:
        warnings.warn("Results obtained with less than %g "
                          "decimal digits of accuracy"
                          % 2, RuntimeWarning)

    if even:
        if N%2 == 0:
            results[:N//2] = np.flip(results[N//2:])
        else:
            results[:N//2] = np.flip(results[N//2+1:])
        return results
    
    return results[N//2:]


In [ ]:
#import warnings
#warnings.filterwarnings("ignore")

### Invert function ###

### Varying k ###

N = 2
c = 2
ks = np.linspace(1/2, 4, N)[1:]
print("c =", c, "k =", ks)

L = 10
x = np.linspace(-L, L, 10_001)

for k in ks:
    
    plt.plot(x, eta_f_c_k(x, c, k, eps_smooth=1e-6), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
             marker="none")

    plt.plot(x, eta_f_c_k(x, c, k), label=r"$(c, k)={}$ non-smooth".format((round(c, 3), round(k, 3))),
             marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")

#plt.savefig(path_save+"eta_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_ks.pgf", dpi=300, bbox_inches = "tight")

plt.xlim(-.1, .1)
plt.ylim(0.6, 1)
plt.show()

### $u_{c, \kappa}$

In [ ]:
## Solitons in variable u

def rho_c_k(x, eta):
    output = np.sqrt(1 - eta)
    return output
    

def theta_c_k(x, c, eta):
    
    ## Function to integrate
    integrand = eta/(1 - eta)
    
    N = x.size
    m = N//2
    xpos = x[m:]
    theta = np.zeros(N)
    
    dx = xpos[1] - xpos[0]

    for j in range(xpos.size-1):
        
        theta[j+1+m] = theta[j+m] + c/2 * (dx/2 * (integrand[j+m] + integrand[j+m+1]))
        
    if N%2 == 0:
        theta[:m] = -np.flip(theta[m:])

    else:
        theta[:m] = -np.flip(theta[m+1:])
    
    return theta


def u_c_k(x, c, eta):
    i = 1j
    
    if c == 0:
        output = np.sign(x) * np.sqrt(1 - eta)
        return output
    
    rho = rho_c_k(x, eta)
    theta = theta_c_k(x, c, eta)
    return rho * np.exp(i*theta)

def norm(x):
    return np.abs(x)**2

In [ ]:
N = 6
c = 2
ks = np.linspace(1/2, 4, N)[1:]
print("c =", c, "k =", ks)

L = 10
x = np.linspace(-L, L, 1_001)

for k in ks:
    #u_cuspon, x = soliton(c, k, soliton_type="cuspon", L=L, N=1001)

    eta = eta_f_c_k(x, c, k)
    u_cuspon = u_c_k(x, c, eta)

    plt.plot(x, norm(u_cuspon), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
             marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u_{c, k}|^2$")

#plt.savefig(path_save+"eta_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()


In [ ]:
c = 2
k = 1.2
L = 10

x = np.linspace(-L, L, 10_001)

eta = eta_f_c_k(x, c, k)
u_cuspon = u_c_k(x, c, eta)

In [ ]:
E_D_1_3(c, k, -1)

In [ ]:
h = x[1] - x[0]
w = u_cuspon

e_0 = h * np.sum(1/2 * norm(np.gradient(w, h)) + 1/4 * (1 - norm(w))**2 - k/4 * (np.gradient(norm(w), h))**2)
e_0

In [ ]:
N = 6
c = 2
ks = np.linspace(1/2, 4, N)[1:]
print("c =", c, "k =", ks)

L = 10
x = np.linspace(-L, L, 1_001)

for k in ks:
    u_cuspon, x = soliton(c, k, soliton_type="cuspon", L=L, N=1001)

    plt.plot(x, norm(u_cuspon), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
             marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u_{c, k}|^2$")

#plt.savefig(path_save+"eta_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()


In [ ]:
c = 2
k = 1.2
L = 50
u_cuspon, x = soliton(c, k, soliton_type="cuspon", L=L, N=20_001)

In [ ]:
E_D_1_3(c, k, -1)

In [ ]:
h = x[1] - x[0]
w = u_cuspon

e_0 = h * np.sum(1/2 * norm(np.gradient(w, h)) + 1/4 * (1 - norm(w))**2 - k/4 * (np.gradient(norm(w), h))**2)
e_0

In [ ]:
N = 6
cs = np.linspace(1.5, 5, N)[1:]
k = 0.6
print("c =", c, "k =", ks)

L = 2
x = np.linspace(-L, L, 1_001)

for c in cs:
    u_cuspon, x = soliton(c, k, soliton_type="cuspon", L=L, N=1001)

    plt.plot(x, norm(u_cuspon), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
             marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u_{c, k}|^2$")

#plt.savefig(path_save+"eta_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
c = 5
k = 0.6
N = 10_001
L = 5
u, x = soliton(c, k, soliton_type="cuspon", L=L, N=N)
h = x[1] - x[0]

#u_prime = np.gradient(u, h)
#u_prime_prime = np.gradient(u_prime, h)
#norm_prime = np.gradient(1-norm(u), h)
#norm_prime_prime = np.gradient(norm_prime, h)
i = 1j


In [ ]:
u_prime = np.gradient(u, h)
u_prime_prime = np.gradient(u_prime, h)
norm_prime = np.gradient(1-np.abs(u)**2, h)
norm_prime_prime = np.gradient(norm_prime, h)
i = 1j

ODE_u = i * c * u_prime + u_prime_prime + u * (1 - np.abs(u)**2) + k * u * norm_prime_prime
plt.plot(x, np.abs(ODE_u), ls="none", ms=1)
plt.xlim(-0.1, 0.1)

In [ ]:
eta = 1 - norm(u(x))
eta_prime = np.gradient(eta, h)

#ODE_u = i * c * u_prime + u_prime_prime + u * (1 - norm(u)) + k * u * norm_prime_prime
ODE_eta = (1 - 2 * k + 2 * k * eta) * eta_prime**2 - eta**2 * (2 - c**2 - 2 * eta)

plt.plot(x, ODE_eta, ls="none", ms=1)
plt.xlim(-0.01, 0.01)

# Anti-Dark Solitons in $D_3$

In [ ]:
path_save = "Solitons/Antidark/"
# Function to invert
def H_c_k(y, c, k):
    
    # Domain
    ymin = 1-c*c/2
    ymax = 0
    
    y = np.asarray(y)
    
    valid_mask = (y >= ymin) & (y < ymax)
    
    # Assign infinite to values outside the domain
    val = np.full_like(y, -np.inf, dtype=np.float64)
    val[y > ymax] = np.inf
    
    def h_c_k(y):
        root = np.sqrt((2-c*c-2*y)/(1-2*k+2*k*y))
        sum_1 = 2 * np.sqrt(k) * np.arctan(np.sqrt(k) * root)
        sum_2 = 2 * np.sqrt((1-2*k)/(2-c*c)) * np.arctanh(np.sqrt((1-2*k)/(2-c*c)) * root)
        output = sum_1 + sum_2
        return output
    
    val[valid_mask] = h_c_k(y[valid_mask])
    
    return val

In [ ]:
### Varying k ###

N = 5
c = 2
ks = np.linspace(1/2, 5, N)[1:]
#print("c =", c, "k =", ks)

y_to_plot = np.linspace(1-c*c/2, 0, 1_000, endpoint=False)

for k in ks:
    plt.plot(y_to_plot, H_c_k(y_to_plot, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()
    
plt.xlabel(r"$y$")
plt.ylabel(r"$H_{c, k}(y)$")

#plt.savefig(path_save+"Func_to_inv_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying c ###

N = 5
cs = np.linspace(1+np.sqrt(2), 3, N)[1:]
k = 2
#print("c =", cs, "k =", k)

for c in cs:
    y_to_plot = np.linspace(1-c*c/2, 0, 1_000, endpoint=False)
    plt.plot(y_to_plot, H_c_k(y_to_plot, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()
    
plt.xlabel(r"$y$")
plt.ylabel(r"$H_{c, k}(y)$")
#plt.savefig(path_save+"Func_to_inv_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Func_to_inv_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## Antidark soliton

In [ ]:
## eta_c_k generator

def eta_H_c_k(x, c, k, even=True):
    
    # Domain
    ymin = 1-c*c/2
    ymax = 0
    
    ref1, ref2 = _get_valid_refpoints(ymin, ymax)
    
    min_kwargs = {}
    min_kwargs['bracket'] = (ref1, ref2)
    min_kwargs['tol'] = 1.48e-12
    min_kwargs['method'] = 'Golden'
    
    x = np.asarray(x)
    N = x.size
    m = N//2
    
    if N == 1:
        xpos = x
        
    else:
        xpos = x[N//2:]
    
    H_to_inv = lambda y: H_c_k(y, c, k)
    
    results = np.full_like(x, np.nan, dtype=np.float64)
    
    for j in range(xpos.size):
        
        if H_to_inv(ymax) == xpos[j]:
            results[N//2+j] = ymax
        
        if H_to_inv(ymin) == xpos[j]:
            results[N//2+j] = ymin
        
        optimizer = (lambda x, j=j,
                        bounded_f=H_to_inv: (np.abs(H_to_inv(x) - xpos[j]))**2)
        
        result = minimize_scalar(optimizer, **min_kwargs)
        results[N//2+j] = result.x
    
    if N%2 == 0:
        results[:m] = np.flip(results[m:])
    else:
        results[:m] = np.flip(results[m+1:])
        
    return results

In [ ]:
import warnings
#warnings.filterwarnings("ignore")

### Invert function ###

### Varying k ###

N = 5
c = 2
ks = np.linspace(1/2, 5, N)[1:]
#print("c =", c, "k =", ks)

L = 20
x = np.linspace(-L, L, 1_000)


for k in ks:
    
    plt.plot(x, eta_H_c_k(x, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")

#plt.savefig(path_save+"eta_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying c ###

N = 5
cs = np.linspace(1+np.sqrt(2), 4, N)[1:]
k = 5
#print("c =", cs, "k =", k)

L = 10
x = np.linspace(-L, L, 1_000)
xs = x[len(x)//2:]

for c in cs:
    
    plt.plot(x, eta_H_c_k(x, c, k), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$x$")
plt.ylabel(r"$\eta_{c, k}(x)$")

#plt.savefig(path_save+"eta_cs.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"eta_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

## Checking ODE $\eta$ 

In [ ]:
c = 2
k = 5

L = 40
x = np.linspace(-L, L, 10_001)
h = x[1] - x[0]

eta = eta_H_c_k(x, c, k)
eta_prime = np.gradient(eta, h)
eta_prime_prime = np.gradient(eta_prime, h)

ODE = (1 - 2 * k + 2 * k * eta) * eta_prime_prime + k * eta_prime**2 + 3 * eta**2 - (2-c**2) * eta
ODE_2 = (1 - 2 * k + 2 * k * eta) * eta_prime**2 - (2 - c**2 - 2 * eta) * eta**2

In [ ]:
plt.plot(x, ODE, ".")
plt.plot(x, ODE_2, ".")

In [ ]:
c, k = 2, 5
print("c = ", c, "k = ", k)
Ns = np.array([2**(j) for j in range(2, 11)])
hs = []
errors = [] #np.zeros(len(Ns))
xs = []
L = 20

for j in range(len(Ns)):
    N = Ns[j]
    x = np.linspace(-L, L, N)
    eta = eta_H_c_k(x, c, k, even=False)
    
    x_pos = x[N//2:]
    h = x_pos[1] - x_pos[0]
    hs += [h]
    F_eta = H_c_k(eta, c, k)
    error = np.abs(x_pos - F_eta)
    
    errors += [error]
    xs += [x_pos]
    
hs = np.array(hs)

In [ ]:
for j in range(len(errors)):
    plt.plot(xs[j], errors[j], ".", label="Ns = {}".format(Ns[j]))
#plt.plot(hs, hs**-0.8 * np.exp(-28))
plt.legend(fontsize=10)
plt.hlines(1.48e-12, 0, 20)
plt.yscale("log")
#plt.xscale("log")
plt.show()

In [ ]:
print("c = ", c, "k = ", k)
Ns = np.array([2**(j) for j in range(2, 11)])
hs = []
errors = [] #np.zeros(len(Ns))
ys = []
L = 20

for j in range(len(Ns)):
    N = Ns[j]
    y = np.linspace(1-c*c/2, 0, N, endpoint=False)
    F_ = H_c_k(y, c, k)
    F_ = np.append(np.flip(F_), F_)
    eta = eta_H_c_k(F_, c, k, even=False)
    error = np.abs(y - eta)

    errors += [error]
    ys += [y]

hs = np.array(hs)

In [ ]:
for j in range(len(errors)):
    plt.plot(ys[j], errors[j], ".", label="Ns = {}".format(Ns[j]))
#plt.plot(hs, hs**-0.8 * np.exp(-28))
plt.legend(fontsize=10)
#plt.hlines(1.48e-12, 0, 5)
#plt.yscale("log")
#plt.xscale("log")
plt.show()

## $u$ variable

In [ ]:
### Varying k ###

N = 5
c = 2
ks = np.linspace(1/2, 5, N)[1:]
#print("c =", c, "k =", ks)


L = 20
x = np.linspace(-L, L, 2_001)


for k in ks:
    eta = eta_H_c_k(x, c, k)
    plt.plot(x, norm(u_c_k(x, c, eta)), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u_{c, k}(x)|^2$")

#plt.savefig(path_save+"D3_antidark_soliton_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"D3_antidark_soliton_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
c = 2
k = 2
L = 20

x = np.linspace(-L, L, 10_001)

eta = eta_H_c_k(x, c, k)
u_antidark = u_c_k(x, c, eta)

In [ ]:
E_D_1_3(c, k, -1)

In [ ]:
h = x[1] - x[0]
w = u_antidark

e_0 = h * np.sum(1/2 * norm(np.gradient(w, h)) + 1/4 * (1 - norm(w))**2 - k/4 * (np.gradient(norm(w), h))**2)
e_0

In [ ]:
### Varying c ###

N = 5
cs = np.linspace(1+np.sqrt(2), 4, N)[1:]
k = 2
print("c =", cs, "k =", k)

L = 10
x = np.linspace(-L, L, 2001)

for c in cs:

    eta = eta_H_c_k(x, c, k)
    plt.plot(x, norm(u_c_k(x, c, eta)), label=r"$(c, k)={}$".format((round(c, 3), round(k, 3))),
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u_{c, k}(x)|^2$")

plt.savefig(path_save+"D3_antidark_soliton_cs.pdf", dpi=300, bbox_inches = "tight")
plt.savefig(path_save+"D3_antidark_soliton_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()


# Bright Solitons

In [ ]:
path_save = "Solitons/bright/"

In [ ]:
# Function to invert

    
def Fb_c_k(y, w, k):
    
    # Domain
    ymin = 0
    ymax = np.sqrt(2*w)
    
    y = np.asarray(y)
    
    valid_mask = (y > ymin) & (y < ymax)
    
    # Assign infinite to values outside the domain
    val = np.full_like(y, -np.inf, dtype=np.float64)
    val[y > ymax] = np.inf

    def f_w_k(y):
        root = np.sqrt((2*w - y*y)/(1+2*k*y*y))
        sum_1 = 1 / np.sqrt(w) * np.arctanh(1/np.sqrt(2*w) * root)
        sum_2 = 2 * np.sqrt(k) * np.arctan(np.sqrt(2*k) * root)
        output = sum_1 + sum_2

        return output
    
    val[valid_mask] = f_w_k(y[valid_mask])
    
    return val

In [ ]:
### Varying k ###

N = 5
w = 1
ks = np.linspace(0, 10, N)[1:]
#print("c =", c, "k =", ks)

y_to_plot = np.linspace(0, np.sqrt(2*w), 1_000, endpoint=False)

for k in ks:
    plt.plot(y_to_plot, Fb_c_k(y_to_plot, w, k), label=r"$(\omega, k)={}$".format((round(w, 3), round(k, 3))),
            marker="none")
    plt.legend()
    
plt.xlabel(r"$y$")
plt.ylabel(r"$F_{\omega, k}(y)$")

plt.savefig(path_save+"Func_to_inv_ks.pdf", dpi=300, bbox_inches = "tight")
plt.savefig(path_save+"Func_to_inv_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying k ###

N = 5
k = 1
ws = np.linspace(0, 4, N)[1:]
#print("c =", c, "k =", ks)

for w in ws:
    y_to_plot = np.linspace(0, np.sqrt(2*w), 1_000, endpoint=False)
    plt.plot(y_to_plot, Fb_c_k(y_to_plot, w, k), label=r"$(\omega, k)={}$".format((round(w, 3), round(k, 3))),
            marker="none")
    plt.legend()
    
plt.xlabel(r"$y$")
plt.ylabel(r"$F_{\omega, k}(y)$")

plt.savefig(path_save+"Func_to_inv_ws.pdf", dpi=300, bbox_inches = "tight")
plt.savefig(path_save+"Func_to_inv_ws.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
## eta_c_k generator

def eta_Fb_c_k(x, w, k, even=True):
    
    # Domain
    ymin = 0
    ymax = np.sqrt(2*w)
    
    ref1, ref2 = _get_valid_refpoints(ymin, ymax)
    
    min_kwargs = {}
    min_kwargs['bracket'] = (ref1, ref2)
    min_kwargs['tol'] = 1.48e-12
    min_kwargs['method'] = 'Golden'
    
    x = np.asarray(x)
    N = x.size
    m = N//2
    
    if N == 1:
        xpos = x
        
    else:
        xpos = x[N//2:]
    
    H_to_inv = lambda y: Fb_c_k(y, w, k)
    
    results = np.full_like(x, np.nan, dtype=np.float64)
    
    for j in range(xpos.size):
        
        if H_to_inv(ymax) == xpos[j]:
            results[N//2+j] = ymax
        
        if H_to_inv(ymin) == xpos[j]:
            results[N//2+j] = ymin
        
        optimizer = (lambda x, j=j,
                        bounded_f=H_to_inv: (np.abs(H_to_inv(x) - xpos[j]))**2)
        
        result = minimize_scalar(optimizer, **min_kwargs)
        results[N//2+j] = result.x
    
    if N%2 == 0:
        results[:m] = np.flip(results[m:])
    else:
        results[:m] = np.flip(results[m+1:])
        
    return results

In [ ]:
### Varying k ###

N = 5
c = 0
w = 1
ks = np.linspace(0, 10, N)[1:]
print("w =", w, "k =", ks)


L = 20
x = np.linspace(-L, L, 2_001)


for k in ks:
    print("w =", w, "k =", k)
    #eta = eta_c_k(x, c, k)
    u_c_k, x = soliton(c, k, w=w, soliton_type="bright", L=L, N=1001)

    plt.plot(x, norm(u_c_k), label=r"$\upkappa = {}$".format(round(k, 3)), #r"$(c, k)=\Big(\frac{\sqrt{2}}{2},$"+str(round(k, 3))+r"$\Big)$",
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u(x)|^2$", rotation='horizontal', ha='right', multialignment='center')
plt.savefig(path_save+"Bright_soliton_ks.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"D_1_dark_soliton_ks.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
### Varying c ###

N = 5
k = 1
ws = np.linspace(0, 4, N)[1:]
c = 0
print("w =", ws, "k =", k)

L = 10
#x = np.linspace(-L, L, 1_000)
#x = np.linspace(-L, L, 2_001)
for j in range(len(ws)):
    w = ws[j]
    #eta = eta_c_k(x, c, k)
    u_c_k, x = soliton(c, k, w=w, soliton_type="bright", L=L, N=1001)
    plt.plot(x, norm(u_c_k), label=r"$\omega = {}$".format(round(w, 3)),#r"$(c, k)=\Big($"+labels[j]+r"$,$"+str(round(k, 3))+r"$\Big)$",
            marker="none")
    plt.legend(fontsize=7)

plt.xlabel(r"$x$")
plt.ylabel(r"$|u(x)|^2$", rotation='horizontal', ha='right', multialignment='center')

plt.savefig(path_save+"Bright_soliton_ws.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"D_1_dark_soliton_cs.pgf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
import warnings
#warnings.filterwarnings("ignore")

### Invert function ###

### Varying k ###

N = 5
w = 1
ks = np.linspace(0, 10, N)[1:]
#print("c =", c, "k =", ks)

L = 20
x = np.linspace(-L, L, 1_000)

for k in ks:
    
    plt.plot(x, norm(eta_Fb_c_k(x, w, k)), label=r"$(\omega, k)={}$".format((round(w, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$x$")
plt.ylabel(r"$|u(x)|^2$")

#plt.savefig(path_save+"Bright_soliton_ks.pdf", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
import warnings
#warnings.filterwarnings("ignore")

### Invert function ###

### Varying k ###

N = 5
k = 1
ws = np.linspace(0, 4, N)[1:]
#print("c =", c, "k =", ks)

L = 10
x = np.linspace(-L, L, 1_000)

for w in ws:
    plt.plot(x, norm(eta_Fb_c_k(x, w, k)), label=r"$(\omega, k)={}$".format((round(w, 3), round(k, 3))),
            marker="none")
    plt.legend()

plt.xlabel(r"$x$")
plt.ylabel(r"$|u|^2$")

#plt.savefig(path_save+"Bright_soliton_ws.pdf", dpi=300, bbox_inches = "tight")
#plt.savefig(path_save+"Bright_soliton_ws.pgf", dpi=300, bbox_inches = "tight")
plt.show()